[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/095.%20The%20Supply%20Chain%20Network%20Design%20under%20Uncertainty/P95-Tier-5_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 95. Supply Chain Network Design under Uncertainty
## Tier 5: Integrated Digital Twin and System-of-Systems Simulation

### Purpose and scope

This notebook builds a small, synthetic digital-twin-style simulation for a supply chain network. It connects simulated sensor readings, simple demand and disruption forecasts, candidate network configurations, a weighted performance score, and a KPI report. Its purpose is to demonstrate how these components can interact in a repeated decision cycle.

The network has facilities, customers, products, and uncertainty scenarios. The simulation produces sensor data and forecasts, then uses a candidate-based heuristic to consider a small set of configuration changes. It does **not** connect to physical IoT devices, ERP or WMS platforms, or external partners. Its performance figures are outputs of this illustrative model, not measured business results.

### Model assumptions

- Facility, customer, product, and sensor data are synthetic. Transportation costs are generated once from the fixed random seed.
- The sensor generator produces scalar synthetic readings. In particular, a GPS-labelled sensor is a signal proxy, not a geographic coordinate.
- Demand forecasts and disruption predictions are stochastic model outputs. They are not trained on historical observations.
- The network configuration consists of open facilities and customer-to-facility assignments. This heuristic does not enforce facility capacity, inventory balance, or service-time constraints.
- The scenario probabilities are stored with the instance for context; the simulation's disruption generator does not sample from those scenarios or use the probabilities in its score.
- The optimizer evaluates the current configuration and a small set of randomly generated alternatives. It is a threshold-based heuristic, not an exact optimization model or a guarantee of a global optimum.

### Solution sequence

1. Construct the network instance and its synthetic transportation costs.
2. Generate sensor readings and flag unusual observations using a rolling statistical rule.
3. Forecast demand and generate potential disruptions.
4. Evaluate the current network and candidate changes using a weighted score.
5. Apply a candidate only when it clears the specified improvement threshold, then record KPIs.
6. Summarize and visualize the simulation results.

### Heuristic score and decision rule

The score combines normalized cost, service level, disruption resilience, flexibility, and sustainability. Higher scores are preferred:

$$S(x,\omega)=0.30C(x)+0.25L(x)+0.20R(x,\omega)+0.15F(x)+0.10U(x),$$

where $x$ is a network configuration, $\omega$ is the predicted disruption set, $C$ is the cost score, $L$ is the service-level score, $R$ is the resilience score, $F$ is the flexibility score, and $U$ is the sustainability score. Each component is scaled to the interval $[0,1]$ in this illustrative model.

Let $\mathcal{N}(x)$ be the alternatives generated from the current configuration. The heuristic selects the highest-scoring candidate that improves on the current score by more than the threshold $\tau=0.05$:

$$x' = \begin{cases}
\displaystyle \arg\max_{y\in\mathcal{N}(x)} S(y,\omega), & \text{if } S(y,\omega)>S(x,\omega)(1+\tau) \text{ for a candidate }y,\\
x, & \text{otherwise.}
\end{cases}$$

The implementation evaluates alternatives one at a time and retains the best qualifying candidate. Its work per decision cycle grows with the number of candidates and the cost of evaluating each configuration; this notebook does not enumerate every possible network configuration.

### What to look for in the results

- KPI time series show how the simulated configuration performs over the chosen horizon.
- Anomaly counts are based on the rolling sensor rule, not on verified operational incidents.
- Optimization counts track configuration changes that the heuristic applies; they are not a measure of optimality.
- Scenario stress-test values are outputs of explicitly stated illustrative formulas, not forecasts validated against real data.

### Illustrative case figures

The problem description gives a separate example of a global electronics manufacturer with 12 facilities, 20 customers, and 8 uncertainty scenarios. It reports 500+ sensors, one-second update cycles, 99.7% uptime, 234 monitored KPIs, 10-times-real-time scenario testing, a 45% improvement in disruption response time, and a 30% reduction in inventory costs. These are contextual case figures, not results produced or verified by this notebook. The synthetic instance below creates fewer than 500 sensors and does not model uptime, KPI breadth, or those business outcomes.

### Why this tier exists

An integrated digital twin aims to connect data, predictions, simulation, and decision support across a supply chain. Compared with a single offline network-design calculation, a repeated simulation can illustrate how changing information might inform successive decisions. The example here demonstrates that integration concept without claiming production-grade real-time synchronization.

**Potential benefits** include more frequent assessment of changing conditions, predictive inputs to planning, and a shared view of network performance. **Practical challenges** include data quality, sensor reliability, system integration, computational requirements, security, and organizational change.

### When this approach may be useful

A digital-twin approach may be worth considering for large or disruption-sensitive supply chains that can support reliable data integration and ongoing model validation. Its value depends on the quality of the data, the suitability of the models, and whether decision makers can act on the resulting information.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A fixed seed makes the synthetic instance and random simulation inputs reproducible.
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class DigitalTwinInstance:
    """Synthetic data describing the supply chain used in the simulation."""

    facilities: List[int]
    customers: List[int]
    products: List[int]
    scenarios: List[int]
    fixed_costs: List[float]
    capacities: List[float]
    transport_costs: Dict[Tuple[int, int, int], float]
    scenario_probs: List[float]
    iot_sensors: List[str]
    update_frequency: float


def create_digital_twin_instance() -> DigitalTwinInstance:
    """Create the synthetic network instance and its sensor identifiers."""
    facilities = list(range(12))
    customers = list(range(20))
    products = list(range(5))
    scenarios = list(range(8))

    fixed_costs = [
        180, 220, 160, 250, 200, 170,
        210, 190, 240, 150, 205, 215,
    ]
    capacities = [
        150, 180, 130, 200, 160, 140,
        170, 150, 190, 120, 155, 165,
    ]

    # A fixed synthetic cost table lets candidate configurations be compared consistently.
    transport_costs = {}
    for facility in facilities:
        for customer in customers:
            for product in products:
                distance_factor = abs(facility - customer // 2) * 0.8
                product_factor = 1.0 + product * 0.15
                base_cost = 2.5 + distance_factor + product_factor
                transport_costs[(facility, customer, product)] = max(
                    1.8, base_cost + np.random.normal(0, 0.3)
                )

    scenario_probs = [0.15, 0.20, 0.25, 0.15, 0.10, 0.05, 0.05, 0.05]

    iot_sensors = (
        [f"FACILITY_{i}_TEMPERATURE" for i in facilities]
        + [f"FACILITY_{i}_HUMIDITY" for i in facilities]
        + [f"FACILITY_{i}_INVENTORY_LEVEL" for i in facilities]
        + [f"VEHICLE_{j}_GPS" for j in range(50)]
        + [f"VEHICLE_{j}_TEMP" for j in range(50)]
        + [f"CUSTOMER_{k}_DEMAND_RATE" for k in customers]
        + [f"WEATHER_{region}_CONDITIONS" for region in ["NORTH", "SOUTH", "EAST", "WEST"]]
    )

    if not np.isclose(sum(scenario_probs), 1.0):
        raise ValueError("Scenario probabilities must sum to 1.")

    return DigitalTwinInstance(
        facilities=facilities,
        customers=customers,
        products=products,
        scenarios=scenarios,
        fixed_costs=fixed_costs,
        capacities=capacities,
        transport_costs=transport_costs,
        scenario_probs=scenario_probs,
        iot_sensors=iot_sensors,
        update_frequency=1.0,
    )


digital_twin_instance = create_digital_twin_instance()

print("Synthetic digital-twin instance")
print(f"Facilities: {len(digital_twin_instance.facilities)}")
print(f"Customers: {len(digital_twin_instance.customers)}")
print(f"Products: {len(digital_twin_instance.products)}")
print(f"Uncertainty scenarios: {len(digital_twin_instance.scenarios)}")
print(f"Synthetic sensor identifiers: {len(digital_twin_instance.iot_sensors)}")
print(f"Nominal sensor update interval: {digital_twin_instance.update_frequency:.1f} second")
print("Sample sensor identifiers:")
for sensor in digital_twin_instance.iot_sensors[:10]:
    print(f"  {sensor}")

In [ ]:
class AnomalyDetector:
    """Flag a reading when it is far from its recent historical values."""

    def __init__(self, threshold: float = 3.0):
        self.threshold = threshold

    def detect_statistical_anomaly(self, readings: List[float]) -> bool:
        """Apply a three-sigma-style rule when enough readings are available."""
        if len(readings) < 10:
            return False

        current = readings[-1]
        historical = readings[:-1]
        mean_historical = float(np.mean(historical))
        std_historical = float(np.std(historical))

        if std_historical == 0:
            return False

        z_score = abs(current - mean_historical) / std_historical
        return z_score > self.threshold


class IoTDataGenerator:
    """Generate synthetic scalar readings for the instance's sensor identifiers."""

    def __init__(self, instance: DigitalTwinInstance):
        self.instance = instance
        self.sensor_data: Dict[str, float] = {}
        self.data_history: Dict[str, List[Tuple[float, float]]] = {}
        self.anomaly_detector = AnomalyDetector()
        self.baseline_values = self._initialize_baseline_values()

    def _initialize_baseline_values(self) -> Dict[str, float]:
        """Set illustrative baseline readings for each sensor category."""
        baselines = {}

        for facility in self.instance.facilities:
            baselines[f"FACILITY_{facility}_TEMPERATURE"] = 22.0
            baselines[f"FACILITY_{facility}_HUMIDITY"] = 45.0
            baselines[f"FACILITY_{facility}_INVENTORY_LEVEL"] = 100.0

        for vehicle in range(50):
            # GPS-labelled values are scalar proxies here, not latitude-longitude coordinates.
            baselines[f"VEHICLE_{vehicle}_GPS"] = 0.0
            baselines[f"VEHICLE_{vehicle}_TEMP"] = 20.0

        for customer in self.instance.customers:
            baselines[f"CUSTOMER_{customer}_DEMAND_RATE"] = 10.0

        for region in ["NORTH", "SOUTH", "EAST", "WEST"]:
            baselines[f"WEATHER_{region}_CONDITIONS"] = 1.0

        return baselines

    def generate_sensor_reading(self, sensor_id: str, timestamp: float) -> float:
        """Generate a reading with a smooth time pattern, noise, and occasional spikes."""
        if sensor_id not in self.baseline_values:
            raise KeyError(f"Unknown sensor identifier: {sensor_id}")

        baseline = self.baseline_values[sensor_id]
        time_variation = np.sin(timestamp / 3600.0) * 2.0
        noise = np.random.normal(0, 0.5)
        anomaly = np.random.normal(0, 5.0) if np.random.random() < 0.05 else 0.0
        reading = baseline + time_variation + noise + anomaly

        if "TEMPERATURE" in sensor_id or "TEMP" in sensor_id:
            reading = np.clip(reading, -10, 50)
        elif "HUMIDITY" in sensor_id:
            reading = np.clip(reading, 0, 100)
        elif "INVENTORY" in sensor_id:
            reading = np.clip(reading, 0, 200)
        elif "DEMAND" in sensor_id:
            reading = np.clip(reading, 0, 50)
        elif "WEATHER" in sensor_id:
            reading = np.clip(reading, 0, 2)

        return float(reading)

    def update_all_sensors(self, timestamp: float) -> Dict[str, float]:
        """Create and retain one reading per sensor for the given simulation time."""
        current_readings = {}

        for sensor_id in self.instance.iot_sensors:
            reading = self.generate_sensor_reading(sensor_id, timestamp)
            current_readings[sensor_id] = reading
            history = self.data_history.setdefault(sensor_id, [])
            history.append((timestamp, reading))

            # A bounded history limits memory use while preserving recent context.
            if len(history) > 100:
                history.pop(0)

        self.sensor_data = current_readings
        return current_readings

    def detect_anomalies(self) -> List[Dict]:
        """Return current readings that exceed the rolling statistical threshold."""
        anomalies = []

        for sensor_id, reading in self.sensor_data.items():
            history = self.data_history.get(sensor_id, [])
            if len(history) < 10:
                continue

            recent_readings = [value for _, value in history[-10:]]
            if self.anomaly_detector.detect_statistical_anomaly(recent_readings):
                mean_reading = float(np.mean(recent_readings[:-1]))
                std_reading = float(np.std(recent_readings[:-1]))
                z_score = abs(reading - mean_reading) / std_reading if std_reading > 0 else 0.0
                anomalies.append(
                    {
                        "sensor_id": sensor_id,
                        "reading": reading,
                        "expected_range": (
                            mean_reading - 2 * std_reading,
                            mean_reading + 2 * std_reading,
                        ),
                        "severity": "high" if z_score > 4 else "medium",
                        "timestamp": history[-1][0],
                    }
                )

        return anomalies


print("Sensor generation and statistical anomaly detection are ready.")

In [ ]:
class PredictiveAnalytics:
    """Generate illustrative demand forecasts and disruption predictions."""

    def __init__(self, instance: DigitalTwinInstance):
        self.instance = instance
        self.demand_models = {}
        self.disruption_models = {}
        self.performance_models = {}
        self._initialize_demand_models()
        self._initialize_disruption_models()

    def _initialize_demand_models(self) -> None:
        """Assign synthetic trend and seasonal parameters to each customer-product pair."""
        for customer in self.instance.customers:
            for product in self.instance.products:
                self.demand_models[(customer, product)] = {
                    "trend": np.random.uniform(-0.5, 0.5),
                    "seasonal_amplitude": np.random.uniform(2, 8),
                    "seasonal_phase": np.random.uniform(0, 2 * np.pi),
                    "base_demand": np.random.uniform(5, 15),
                    "noise_level": np.random.uniform(0.5, 2.0),
                }

    def _initialize_disruption_models(self) -> None:
        """Set illustrative occurrence rates and duration parameters by disruption type."""
        disruption_types = ["weather", "traffic", "facility", "supplier", "demand"]
        for disruption_type in disruption_types:
            self.disruption_models[disruption_type] = {
                "base_probability": np.random.uniform(0.01, 0.05),
                "severity_distribution": "exponential",
                "duration_mean": np.random.uniform(2, 8),
                "recovery_time_mean": np.random.uniform(1, 4),
            }

    def forecast_demand(
        self,
        customer: int,
        product: int,
        horizon: int = 24,
        timestamp: float = 0.0,
    ) -> List[float]:
        """Forecast hourly demand using a trend, daily seasonal pattern, and noise."""
        model = self.demand_models[(customer, product)]
        forecast = []

        for hour_ahead in range(horizon):
            trend = model["trend"] * hour_ahead
            seasonal = model["seasonal_amplitude"] * np.sin(
                2 * np.pi * (timestamp + hour_ahead * 3600) / 86400
                + model["seasonal_phase"]
            )
            noise = np.random.normal(0, model["noise_level"])
            demand = max(0.0, model["base_demand"] + trend + seasonal + noise)
            forecast.append(float(demand))

        return forecast

    def predict_disruptions(self, time_horizon: int = 24) -> List[Dict]:
        """Generate a synthetic set of possible disruptions over a time horizon in hours."""
        predicted_disruptions = []

        for disruption_type, model in self.disruption_models.items():
            expected_count = model["base_probability"] * time_horizon
            num_disruptions = np.random.poisson(expected_count)

            for _ in range(num_disruptions):
                severity = min(1.0, float(np.random.exponential(scale=1.0)))
                predicted_disruptions.append(
                    {
                        "type": disruption_type,
                        "start_time": float(np.random.uniform(0, time_horizon)),
                        "duration": float(np.random.exponential(scale=model["duration_mean"])),
                        "severity": severity,
                        "recovery_time": float(
                            np.random.exponential(scale=model["recovery_time_mean"])
                        ),
                        "impact_area": self._get_impact_area(disruption_type),
                        "confidence": float(np.random.uniform(0.6, 0.9)),
                    }
                )

        predicted_disruptions.sort(key=lambda item: item["start_time"])
        return predicted_disruptions

    def _get_impact_area(self, disruption_type: str) -> str:
        """Describe the broad operational area associated with a disruption type."""
        impact_areas = {
            "weather": "Transportation routes",
            "traffic": "Urban deliveries",
            "facility": "Facility operations",
            "supplier": "Inbound supply",
            "demand": "Customer service levels",
        }
        return impact_areas.get(disruption_type, "General operations")

    def predict_network_performance(
        self,
        current_config: Dict,
        predicted_disruptions: List[Dict],
    ) -> Dict[str, float]:
        """Apply simple disruption multipliers to illustrative baseline KPIs."""
        base_metrics = {
            "service_level": 0.95,
            "inventory_turns": 12.0,
            "transportation_cost": 1000.0,
            "facility_utilization": 0.75,
            "order_fulfillment_time": 24.0,
        }
        impact_factors = {metric: 1.0 for metric in base_metrics}

        for disruption in predicted_disruptions:
            severity = disruption["severity"]
            if disruption["type"] == "weather":
                impact_factors["transportation_cost"] *= 1 + 0.3 * severity
                impact_factors["service_level"] *= 1 - 0.1 * severity
                impact_factors["order_fulfillment_time"] *= 1 + 0.2 * severity
            elif disruption["type"] == "facility":
                impact_factors["facility_utilization"] *= 1 - 0.2 * severity
                impact_factors["service_level"] *= 1 - 0.15 * severity
            elif disruption["type"] == "demand":
                impact_factors["inventory_turns"] *= 1 - 0.1 * severity
                impact_factors["service_level"] *= 1 - 0.05 * severity

        return {
            metric: value * impact_factors[metric]
            for metric, value in base_metrics.items()
        }


print("Synthetic demand and disruption forecasting are ready.")

In [ ]:
class OptimizationEngine:
    """Apply a candidate-based, threshold-improvement heuristic to configurations."""

    def __init__(self, instance: DigitalTwinInstance):
        self.instance = instance
        self.current_config = {}
        self.optimization_history = []
        self.performance_history = []
        self.optimization_frequency = 300
        self.last_optimization = 0
        self.optimization_threshold = 0.05

    def optimize_network_configuration(
        self,
        current_state: Dict,
        predicted_disruptions: List[Dict],
        demand_forecasts: Dict,
    ) -> Dict:
        """Evaluate generated alternatives and return an improving configuration if found."""
        current_performance = self._evaluate_configuration(
            current_state, predicted_disruptions, demand_forecasts
        )
        current_score = current_performance["total_score"]
        best_score = current_score
        best_alternative = None

        for alternative in self._generate_alternatives(current_state):
            performance = self._evaluate_configuration(
                alternative, predicted_disruptions, demand_forecasts
            )
            if performance["total_score"] > best_score:
                best_score = performance["total_score"]
                best_alternative = alternative

        # The threshold prevents the heuristic from applying small score changes.
        applied = best_score > current_score * (1 + self.optimization_threshold)
        improvement = (best_score - current_score) / current_score if current_score else 0.0
        self.optimization_history.append(
            {
                "timestamp": time.time(),
                "current_config": current_state,
                "best_alternative": best_alternative,
                "improvement": improvement,
                "applied": applied,
            }
        )

        return best_alternative if applied and best_alternative is not None else current_state

    def _evaluate_configuration(
        self,
        config: Dict,
        predicted_disruptions: Optional[List[Dict]] = None,
        demand_forecasts: Optional[Dict] = None,
    ) -> Dict[str, float]:
        """Calculate cost and score components for one candidate configuration."""
        predicted_disruptions = predicted_disruptions or []
        demand_forecasts = demand_forecasts or {}

        fixed_cost = sum(
            self.instance.fixed_costs[facility]
            for facility in config.get("open_facilities", [])
        )
        variable_cost = self._calculate_variable_cost(config, demand_forecasts)
        total_cost = fixed_cost + variable_cost
        service_level = self._calculate_service_level(config)
        flexibility = self._calculate_flexibility(config)
        sustainability = self._calculate_sustainability(config)
        resilience = self._calculate_disruption_impact(config, predicted_disruptions)

        cost_score = max(0.0, 1.0 - total_cost / 10000.0)
        weights = {
            "cost": 0.30,
            "service_level": 0.25,
            "resilience": 0.20,
            "flexibility": 0.15,
            "sustainability": 0.10,
        }
        total_score = (
            cost_score * weights["cost"]
            + service_level * weights["service_level"]
            + resilience * weights["resilience"]
            + flexibility * weights["flexibility"]
            + sustainability * weights["sustainability"]
        )

        return {
            "total_cost": total_cost,
            "service_level": service_level,
            "flexibility": flexibility,
            "sustainability": sustainability,
            "cost_score": cost_score,
            "disruption_resilience": resilience,
            "total_score": total_score,
        }

    def _generate_alternatives(self, current_config: Dict) -> List[Dict]:
        """Generate the same three types of local configuration alternatives."""
        alternatives = []
        open_facilities = list(current_config.get("open_facilities", []))

        # Candidate 1 opens one randomly selected closed facility.
        closed_facilities = [
            facility
            for facility in self.instance.facilities
            if facility not in open_facilities
        ]
        if closed_facilities:
            new_facility = int(np.random.choice(closed_facilities))
            alternative = current_config.copy()
            alternative["open_facilities"] = open_facilities + [new_facility]
            alternative["customer_assignments"] = dict(
                current_config.get("customer_assignments", {})
            )
            alternatives.append(alternative)

        # Candidate 2 closes one facility when more than three are currently open.
        if len(open_facilities) > 3:
            facility_to_close = int(np.random.choice(open_facilities))
            remaining_facilities = [
                facility for facility in open_facilities
                if facility != facility_to_close
            ]
            alternative = current_config.copy()
            alternative["open_facilities"] = remaining_facilities
            alternative["customer_assignments"] = {
                customer: (
                    facility_to_close
                    if facility == facility_to_close
                    else facility
                )
                for customer, facility in current_config.get(
                    "customer_assignments", {}
                ).items()
            }
            # Customers assigned to a closed facility are moved to an open one.
            for customer, facility in alternative["customer_assignments"].items():
                if facility == facility_to_close:
                    alternative["customer_assignments"][customer] = int(
                        np.random.choice(remaining_facilities)
                    )
            alternatives.append(alternative)

        # Candidate 3 reassigns a sample of customers among the open facilities.
        assignments = dict(current_config.get("customer_assignments", {}))
        if assignments and open_facilities:
            alternative = current_config.copy()
            alternative["customer_assignments"] = assignments
            customer_ids = list(assignments)
            sample_size = max(1, len(customer_ids) // 5)
            customers_to_reassign = np.random.choice(
                customer_ids, size=min(sample_size, len(customer_ids)), replace=False
            )
            for customer in customers_to_reassign:
                alternative["customer_assignments"][int(customer)] = int(
                    np.random.choice(open_facilities)
                )
            alternatives.append(alternative)

        return alternatives

    def _calculate_variable_cost(self, config: Dict, demand_forecasts: Dict) -> float:
        """Estimate transport cost using forecast means or a fixed fallback demand."""
        variable_cost = 0.0
        assignments = config.get("customer_assignments", {})
        open_facilities = config.get("open_facilities", [])

        for customer, facility in assignments.items():
            if facility not in open_facilities:
                continue
            for product in self.instance.products:
                forecast = demand_forecasts.get((customer, product))
                demand = float(np.mean(forecast)) if forecast else 100.0
                cost = self.instance.transport_costs.get(
                    (facility, customer, product), 5.0
                )
                variable_cost += cost * demand

        return variable_cost

    def _calculate_service_level(self, config: Dict) -> float:
        """Calculate the illustrative service score from the number of open facilities."""
        base_service = 0.9
        num_facilities = len(config.get("open_facilities", []))
        if num_facilities < 4:
            base_service -= 0.1 * (4 - num_facilities)
        if 5 <= num_facilities <= 7:
            base_service += 0.05
        return max(0.7, min(0.99, base_service))

    def _calculate_flexibility(self, config: Dict) -> float:
        """Use the share of open facilities as a simple flexibility proxy."""
        num_facilities = len(config.get("open_facilities", []))
        return min(1.0, num_facilities / len(self.instance.facilities))

    def _calculate_sustainability(self, config: Dict) -> float:
        """Use facility count as a simple, explicitly illustrative sustainability proxy."""
        num_facilities = len(config.get("open_facilities", []))
        sustainability = 1.0 - (num_facilities / len(self.instance.facilities)) * 0.3
        return max(0.5, min(1.0, sustainability))

    def _calculate_disruption_impact(
        self,
        config: Dict,
        predicted_disruptions: List[Dict],
    ) -> float:
        """Reduce the resilience proxy when severe disruptions meet a concentrated network."""
        resilience = 1.0
        num_facilities = len(config.get("open_facilities", []))
        facility_resilience = min(1.0, num_facilities / 8.0)

        for disruption in predicted_disruptions:
            resilience -= disruption["severity"] * 0.1 * (1 - facility_resilience)

        return max(0.3, resilience)


print("Candidate-based heuristic optimization engine is ready.")

In [ ]:
class DigitalTwinSystem:
    """Coordinate synthetic sensing, forecasting, heuristic decisions, and KPI recording."""

    def __init__(self, instance: DigitalTwinInstance):
        self.instance = instance
        self.iot_generator = IoTDataGenerator(instance)
        self.predictive_analytics = PredictiveAnalytics(instance)
        self.optimization_engine = OptimizationEngine(instance)
        self.current_time = 0
        self.current_config = self._initialize_configuration()
        self.system_history = []
        self.kpi_history = {
            "timestamps": [],
            "service_levels": [],
            "costs": [],
            "anomalies": [],
            "optimizations": [],
            "disruptions": [],
        }

    def _initialize_configuration(self) -> Dict:
        """Open a random set of facilities and assign each customer to a nearby one."""
        num_facilities = int(np.random.randint(5, 8))
        open_facilities = [
            int(facility)
            for facility in np.random.choice(
                self.instance.facilities, num_facilities, replace=False
            )
        ]

        customer_assignments = {}
        for customer in self.instance.customers:
            nearest_facility = min(
                open_facilities,
                key=lambda facility: (abs(facility - customer // 2), facility),
            )
            customer_assignments[customer] = nearest_facility

        return {
            "open_facilities": open_facilities,
            "customer_assignments": customer_assignments,
        }

    def run_simulation(
        self,
        duration_hours: int = 24,
        time_step_minutes: int = 5,
    ) -> Dict:
        """Run the repeated sensing, prediction, decision, and KPI cycle."""
        if duration_hours <= 0 or time_step_minutes <= 0:
            raise ValueError("Duration and time step must be positive.")
        if time_step_minutes > 60 or 60 % time_step_minutes != 0:
            raise ValueError("The time step must divide one hour evenly and be at most 60 minutes.")

        print("Starting synthetic digital-twin simulation")
        print(f"Duration: {duration_hours} hours")
        print(f"Time step: {time_step_minutes} minutes")
        print(f"Nominal sensor update interval: {self.instance.update_frequency:.1f} second")

        total_steps = duration_hours * 60 // time_step_minutes
        time_step_seconds = time_step_minutes * 60
        steps_per_hour = 60 // time_step_minutes
        total_anomalies = 0
        total_optimizations = 0
        total_disruptions = 0
        predicted_disruptions = []
        demand_forecasts = {}

        for step in range(total_steps):
            self.current_time = step * time_step_seconds
            self.iot_generator.update_all_sensors(self.current_time)
            anomalies = self.iot_generator.detect_anomalies()
            total_anomalies += len(anomalies)

            # Refresh forecasts hourly; the first refresh occurs at the first step.
            if step % steps_per_hour == 0:
                predicted_disruptions = self.predictive_analytics.predict_disruptions(24)
                total_disruptions += len(predicted_disruptions)
                demand_forecasts = {}
                for customer in self.instance.customers[:3]:
                    for product in self.instance.products[:2]:
                        demand_forecasts[(customer, product)] = (
                            self.predictive_analytics.forecast_demand(
                                customer,
                                product,
                                horizon=24,
                                timestamp=self.current_time,
                            )
                        )

            # Reconsider the network at the configured five-minute interval.
            optimization_steps = max(
                1,
                self.optimization_engine.optimization_frequency // time_step_seconds,
            )
            if step % optimization_steps == 0:
                new_config = self.optimization_engine.optimize_network_configuration(
                    self.current_config, predicted_disruptions, demand_forecasts
                )
                if new_config != self.current_config:
                    self.current_config = new_config
                    total_optimizations += 1

            performance = self.optimization_engine._evaluate_configuration(
                self.current_config, predicted_disruptions, demand_forecasts
            )

            if step % steps_per_hour == 0:
                self.kpi_history["timestamps"].append(self.current_time / 3600.0)
                self.kpi_history["service_levels"].append(performance["service_level"])
                self.kpi_history["costs"].append(performance["total_cost"])
                self.kpi_history["anomalies"].append(len(anomalies))
                self.kpi_history["optimizations"].append(total_optimizations)
                self.kpi_history["disruptions"].append(len(predicted_disruptions))

            if step % max(1, total_steps // 10) == 0:
                progress = 100.0 * step / total_steps
                print(f"Simulation progress: {progress:.0f}%")

        print("Simulation completed")
        print(f"Total anomalous readings flagged: {total_anomalies}")
        print(f"Configuration changes applied: {total_optimizations}")
        print(f"Disruption predictions generated: {total_disruptions}")

        return {
            "duration_hours": duration_hours,
            "total_anomalies": total_anomalies,
            "total_optimizations": total_optimizations,
            "total_disruptions": total_disruptions,
            "final_config": self.current_config,
            "kpi_history": self.kpi_history,
        }

    def generate_comprehensive_report(self, simulation_results: Dict) -> Dict:
        """Summarize simulated KPIs without treating assumptions as measured outcomes."""
        kpi_data = simulation_results["kpi_history"]
        if not kpi_data["timestamps"]:
            raise ValueError("The simulation must record at least one KPI observation.")

        service_levels = kpi_data["service_levels"]
        costs = kpi_data["costs"]
        service_improvement = service_levels[-1] - service_levels[0]
        cost_reduction = (
            (costs[0] - costs[-1]) / costs[0] if costs[0] else 0.0
        )
        duration = simulation_results["duration_hours"]

        return {
            "executive_summary": {
                "simulation_duration_hours": duration,
                "kpi_observations": len(kpi_data["timestamps"]),
                "average_service_level": float(np.mean(service_levels)),
                "average_network_cost": float(np.mean(costs)),
                "configuration_changes_applied": simulation_results["total_optimizations"],
                "anomalous_readings_flagged": simulation_results["total_anomalies"],
                "disruption_predictions_generated": simulation_results["total_disruptions"],
            },
            "performance_metrics": {
                "service_level_change": float(service_improvement),
                "network_cost_reduction_fraction": float(cost_reduction),
                "configuration_changes_per_kpi_observation": (
                    simulation_results["total_optimizations"] / len(kpi_data["timestamps"])
                ),
                "flagged_readings_per_kpi_observation": (
                    simulation_results["total_anomalies"] / len(kpi_data["timestamps"])
                ),
            },
            "system_capabilities": {
                "synthetic_sensor_identifiers": len(self.instance.iot_sensors),
                "nominal_sensor_update_interval_seconds": self.instance.update_frequency,
                "synthetic_demand_forecasting": True,
                "synthetic_disruption_prediction": True,
                "candidate_based_configuration_heuristic": True,
            },
        }


print("Digital-twin simulation coordinator is ready.")

In [ ]:
digital_twin = DigitalTwinSystem(digital_twin_instance)
simulation_results = digital_twin.run_simulation(
    duration_hours=24,
    time_step_minutes=5,
)
comprehensive_report = digital_twin.generate_comprehensive_report(simulation_results)

print("\nDigital-twin simulation report")
for section_name, section in comprehensive_report.items():
    print(f"\n{section_name.replace('_', ' ').title()}")
    for key, value in section.items():
        print(f"  {key.replace('_', ' ')}: {value}")

final_config = simulation_results["final_config"]
final_kpis = simulation_results["kpi_history"]
assert set(final_config["open_facilities"]).issubset(digital_twin_instance.facilities)
assert set(final_config["customer_assignments"]) == set(digital_twin_instance.customers)
assert all(
    facility in final_config["open_facilities"]
    for facility in final_config["customer_assignments"].values()
)
assert len(final_kpis["timestamps"]) == len(final_kpis["costs"])

print("\nFinal network configuration")
print(f"  Open facilities: {sorted(final_config['open_facilities'])}")
print(f"  Customers assigned: {len(final_config['customer_assignments'])}")
print(f"  Final recorded network cost: {final_kpis['costs'][-1]:.2f} cost units")
print(f"  Final recorded service-level score: {final_kpis['service_levels'][-1]:.3f}")

In [ ]:
def visualize_digital_twin_performance(
    simulation_results: Dict,
    comprehensive_report: Dict,
) -> None:
    """Plot recorded simulation KPIs and clearly label illustrative metrics."""
    kpi_data = simulation_results["kpi_history"]
    timestamps = kpi_data["timestamps"]
    if not timestamps:
        raise ValueError("There are no KPI observations to visualize.")

    figure, axes = plt.subplots(2, 2, figsize=(15, 10))
    figure.suptitle("Synthetic Digital-Twin Simulation Dashboard", fontsize=15)

    service_axis = axes[0, 0]
    cost_axis = service_axis.twinx()
    service_line = service_axis.plot(
        timestamps, kpi_data["service_levels"], color="tab:blue", label="Service-level score"
    )
    cost_line = cost_axis.plot(
        timestamps, kpi_data["costs"], color="tab:red", label="Network cost"
    )
    service_axis.set_xlabel("Simulation time (hours)")
    service_axis.set_ylabel("Service-level score (0–1)", color="tab:blue")
    cost_axis.set_ylabel("Network cost (cost units)", color="tab:red")
    service_axis.set_title("Service Level and Network Cost")
    service_axis.grid(True, alpha=0.3)
    lines = service_line + cost_line
    service_axis.legend(lines, [line.get_label() for line in lines], loc="best")

    activity_axis = axes[0, 1]
    activity_axis.bar(
        timestamps,
        kpi_data["anomalies"],
        alpha=0.65,
        color="tab:orange",
        label="Readings flagged in hour",
    )
    activity_axis.plot(
        timestamps,
        kpi_data["optimizations"],
        color="tab:green",
        marker="o",
        label="Cumulative configuration changes",
    )
    activity_axis.set_xlabel("Simulation time (hours)")
    activity_axis.set_ylabel("Count")
    activity_axis.set_title("Detection and Configuration Activity")
    activity_axis.legend()
    activity_axis.grid(True, alpha=0.3)

    summary_axis = axes[1, 0]
    summary = comprehensive_report["executive_summary"]
    summary_labels = ["Average service-level score", "Average network cost"]
    summary_values = [summary["average_service_level"], summary["average_network_cost"]]
    bars = summary_axis.bar(summary_labels, summary_values, color=["#4ECDC4", "#45B7D1"])
    summary_axis.set_title("Simulation Summary")
    summary_axis.set_ylabel("Value (different units; see labels)")
    summary_axis.tick_params(axis="x", rotation=12)
    summary_axis.grid(True, alpha=0.3, axis="y")
    for bar, value in zip(bars, summary_values):
        summary_axis.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center",
            va="bottom",
        )

    disruption_axis = axes[1, 1]
    disruption_axis.plot(
        timestamps,
        kpi_data["disruptions"],
        color="tab:purple",
        marker="o",
        label="Predicted disruptions in refreshed horizon",
    )
    disruption_axis.set_xlabel("Simulation time (hours)")
    disruption_axis.set_ylabel("Predicted disruption count")
    disruption_axis.set_title("Synthetic Disruption Predictions")
    disruption_axis.legend()
    disruption_axis.grid(True, alpha=0.3)

    figure.tight_layout()
    plt.show()


visualize_digital_twin_performance(simulation_results, comprehensive_report)

In [ ]:
def analyze_digital_twin_capabilities(
    instance: DigitalTwinInstance,
    predictive_analytics: PredictiveAnalytics,
) -> Dict[str, Dict]:
    """Demonstrate model capabilities and transparent illustrative stress tests."""
    print("Synthetic digital-twin capability review")
    print("\nData and update assumptions")
    print(f"  Sensor identifiers in the synthetic instance: {len(instance.iot_sensors)}")
    print(f"  Nominal update interval recorded in the instance: {instance.update_frequency} second")
    print("  Sensor readings are generated data, not a live physical data stream.")

    print("\nForecast and decision components")
    print("  Demand forecast horizon used in the simulation: 24 hours")
    print("  Disruption types represented: weather, traffic, facility, supplier, and demand")
    print("  Configuration decisions use a weighted score and a 5% improvement threshold.")
    print("  The candidate generator considers facility opening, facility closing, and reassignment.")

    scenarios = {
        "baseline": {"demand_increase": 0.0, "disruption_probability": 0.02},
        "high_demand": {"demand_increase": 0.3, "disruption_probability": 0.02},
        "high_disruption": {"demand_increase": 0.0, "disruption_probability": 0.08},
        "worst_case": {"demand_increase": 0.3, "disruption_probability": 0.08},
    }
    scenario_results = {}

    # These transparent equations are a stress-test illustration, not a fitted forecast.
    for scenario_name, parameters in scenarios.items():
        demand_impact = parameters["demand_increase"]
        disruption_impact = parameters["disruption_probability"] * 50
        base_service = 0.95
        base_cost = 5000.0
        service_level = max(
            0.7,
            base_service - demand_impact * 0.1 - disruption_impact * 0.002,
        )
        total_cost = base_cost * (
            1 + demand_impact * 0.2 + disruption_impact * 0.01
        )
        scenario_results[scenario_name] = {
            "service_level": service_level,
            "total_cost": total_cost,
            "risk_score": disruption_impact / 10,
        }

    print("\nIllustrative scenario stress test (not empirical performance)")
    for scenario_name, results in scenario_results.items():
        print(
            f"  {scenario_name.replace('_', ' ').title():16s} "
            f"Service score={results['service_level']:.3f}, "
            f"Cost units={results['total_cost']:.0f}, "
            f"Risk proxy={results['risk_score']:.2f}"
        )

    return scenario_results


scenario_analysis = analyze_digital_twin_capabilities(
    digital_twin_instance,
    digital_twin.predictive_analytics,
)

## Interpreting the simulation

The notebook's repeated cycle is the central system-of-systems idea: generated observations inform forecasts, forecasts enter a configuration-scoring step, and the resulting state produces recorded KPIs. The optimizer's role is deliberately limited to evaluating a small neighborhood of the current configuration and applying a change only when the weighted score clears its threshold.

The plots and report describe this synthetic model run. A flagged sensor value is not proof of a real operational anomaly, a predicted disruption is not a confirmed event, and a configuration change is not evidence of a globally optimal network. The service, cost, flexibility, sustainability, and resilience measures are simplified proxies; in particular, the model does not enforce facility capacities even though capacity data are present in the instance.

The separate scenario stress test uses fixed illustrative equations to show how assumptions can be varied and their modeled effects compared. Its values are not generated by the forecasting model and should not be interpreted as validated predictions. Likewise, the case figures in the opening section provide context only; they are not simulated or confirmed here.

A production digital twin would require validated data connections, operationally meaningful constraints and units, model calibration, monitoring of prediction error, and procedures for human review. Those extensions would build on the integration pattern demonstrated here without changing the distinction between model outputs and observed business evidence.